# AValueBaseJ

> Base class containing the core methods of CRLD agents in value space (JAX variant).

In [ ]:
#| default_exp Agents/ValueBaseJ

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import numpy as np
import itertools as it
from functools import partial

import jax
from jax import jit
import jax.numpy as jnp
from typing import Iterable
from fastcore.utils import *

from pyCRLD.Agents.BaseJ import abaseJ
from pyCRLD.Utils.HelpersJ import *

In [ ]:
#| export
class multiagent_epsilongreedy_strategyJ():
    """Multiagent epsilon-greedy strategy."""

    def __init__(self, epsilon_greedys=None, N=None):
        """
        Parameters
        ----------
        epsilon_greedys : iterable or float
            exploration parameter(s) for each agent
        N : int
            number of agents, only allowed if `epsilon_greedys` is a single float
        """
        egiter = hasattr(epsilon_greedys, '__iter__')

        if egiter:  # eps greedy iter, sm not
            self.N = len(epsilon_greedys)  # Number of agents
            assert N is None, "'N' must not be specified when iterable is given"

        else:
            self.N = N  # Number of agents
            assert epsilon_greedys is not None, "epsilon value must be given"
            assert type(epsilon_greedys) is float, 'Confusing parameter input'
            epsilon_greedys = [epsilon_greedys] * self.N

        # exploration values
        self.epsilongreedy_explorations = \
            jnp.array(epsilon_greedys).astype(float)

In [ ]:
#| export
@partial(jit, static_argnums=0)
def action_probabilitiesJ(self: multiagent_epsilongreedy_strategyJ, Qisa):
    """Transform Q values into epsilongreedy policy"""
    n = jnp.newaxis
    Xisa = jnp.zeros_like(Qisa)

    # where are the actions with maximal value?
    WhereMAXisa = Qisa == jnp.max(Qisa, axis=-1, keepdims=True)

    # assign 1-eps probability to max actions
    eps = self.epsilongreedy_explorations
    Xisa += (1 - eps[:, n, n]) * WhereMAXisa / \
        WhereMAXisa.sum(axis=-1, keepdims=True)

    # assign eps probability to all actions
    Qisa_dimensions = jnp.ones_like(Qisa)
    Xisa += eps[:, n, n] * Qisa_dimensions \
        / Qisa_dimensions.sum(axis=-1, keepdims=True)
    return Xisa

# Monkey-patching
multiagent_epsilongreedy_strategyJ.action_probabilities = action_probabilitiesJ

In [ ]:
#| export
@patch
def id(self: multiagent_epsilongreedy_strategyJ
       ) -> str:  # id
    """Returns an identifier to handle simulation runs."""
    id = f"j{self.__class__.__name__}_"
    for i in range(self.N):
        eg = np.array(self.epsilongreedy_explorations[i])
        id += jnp.array_str(eg, precision=5) + '_'

    return id[:-1]

In [ ]:
#| export
class valuebaseJ(abaseJ):
    """
    Base class for deterministic strategy-average independent (multi-agent) reward-prediction temporal-difference reinforcement learning in value space.
    """

    def __init__(self,
                 env,  # An environment object
                 learning_rates: Union[float, Iterable],  # agents' learning rates
                 discount_factors: Union[float, Iterable],  # agents' discount factors
                 strategy_function,  # the strategy function object
                 choice_intensities: Union[float, Iterable] = 1.0,
                 use_prefactor=False,
                 opteinsum=True,
                 **kwargs):

        self.env = env
        Tt = env.T
        assert np.allclose(Tt.sum(-1), 1)
        Rt = env.R
        super().__init__(Tt, Rt, discount_factors, use_prefactor, opteinsum)
        self.F = jnp.array(env.F)

        # learning rates
        self.alpha = make_variable_vectorJ(learning_rates, self.N)

        # strategy function
        assert env.N == strategy_function.N, \
            'Environment and strategy function must have the same number of\
             agents `N`'
        self.strategy_function = strategy_function

        # temporal difference error mirror (without indices)
        self.TDerror = self.RPEisa

In [ ]:
#| export
@partial(jit, static_argnums=0)
def stepJ(self: valuebaseJ,
         Qisa):  # joint state-action values
    """Learning step in value space, given joint state-action values `Qisa`."""
    RPisa = self.TDerror(Qisa)
    Qisa_ = Qisa + self.alpha * RPisa
    return Qisa_, RPisa
valuebaseJ.step = stepJ  # Monkey-patching

In [ ]:
#| export
@patch
def zero_intelligence_values(self: valuebaseJ,
                             value: float = 0.0):  # state-action value
    """Zero-intelligence state-action values."""
    return value * jnp.ones((self.N, self.Z, self.M))

@patch
def random_values(self: valuebaseJ,
                  key: jnp.ndarray = None  # JAX PRNG key
                  ) -> jnp.ndarray:
    """Returns normally distributed random state-action values."""
    if key is not None:
        return jax.random.normal(key, shape=(self.N, self.Z, self.M))
    else:
        return jnp.array(np.random.randn(self.N, self.Z, self.M))

In [ ]:
#| export
def id(self: valuebaseJ
       ) -> str:  # id
    """Returns an identifier to handle simulation runs."""
    envid = self.env.id() + "__"
    agentsid = f"j{self.__class__.__name__}_"\
        + f"{str(self.alpha)}_{str(self.gamma)}_pre{self.use_prefactor}__"
    strategyid = self.strategy_function.id()

    return envid + agentsid + agentsid

## Tests

In [ ]:
from pyCRLD.Environments.SocialDilemmaJ import SocialDilemmaJ

env = SocialDilemmaJ(R=3., T=5., S=0., P=1.)
strat_fn = multiagent_epsilongreedy_strategyJ(epsilon_greedys=0.1, N=2)

# Test epsilon-greedy action probs
Q = jnp.array([[[1.0, 2.0]], [[2.0, 1.0]]])  # shape (2,1,2)
X = strat_fn.action_probabilities(Q)
assert X.shape == (2, 1, 2)
assert jnp.allclose(X.sum(-1), 1.0)

In [ ]:
# Test random_values with JAX key
class _TestVal(valuebaseJ):
    @partial(jit, static_argnums=0)
    def RPEisa(self, Qisa, norm=False):
        return jnp.zeros_like(Qisa)  # dummy

agent = _TestVal(env, learning_rates=0.1, discount_factors=0.9, strategy_function=strat_fn)

key = jax.random.PRNGKey(0)
Qr = agent.random_values(key=key)
assert Qr.shape == (2, 1, 2)

Q0 = agent.zero_intelligence_values()
assert Q0.shape == (2, 1, 2)
assert jnp.all(Q0 == 0.0)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()